# Public coffee-dataset eligibility audit
Audit metadata, annotations, split leakage, and cross-dataset duplicate lineage. **No training and no checkpoint evaluation.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib, os, shutil, subprocess, sys
from pathlib import Path
REPO=Path('/content/coffee-bean-detection')
BRANCH='codex/public-dataset-eligibility-audit'
REMOTE='https://'+'github.com/ediprin/coffee-bean-detection.git'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone_error=''
for attempt in range(1,4):
    clone=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REMOTE,str(REPO)],text=True,capture_output=True)
    if clone.returncode==0: break
    clone_error=(clone.stderr or clone.stdout).strip(); print(f'CLONE ATTEMPT {attempt}/3 GAGAL:',clone_error)
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('GitHub tidak dapat diakses setelah 3 percobaan: '+clone_error)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.analysis.public_dataset_eligibility import audit_public_dataset_registry, extract_audit_archive
print('REPO READY:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
# Isi hanya path export yang benar-benar sudah diekstrak dan arsip aslinya.
# Kode harus sama dengan registry. Kandidat yang belum ada boleh dibiarkan kosong.
ROOTS={
    # 'lulus_v1': Path('/content/lulus-v1'),
    # 'capstone_v1': Path('/content/capstone-v1'),
    # 'niacubilla_v1': Path('/content/niacubilla-v1'),
    # 'coffee_standard_v8': Path('/content/coffee-standard-v8'),
}
ARCHIVES={
    # 'lulus_v1': Path('/content/drive/MyDrive/Coffee_Bean_Detection/bundles/lulus-v1-yolov8.zip'),
    # 'capstone_v1': Path('/content/drive/MyDrive/Coffee_Bean_Detection/bundles/capstone-v1-yolov8.zip'),
    # 'niacubilla_v1': Path('/content/drive/MyDrive/Coffee_Bean_Detection/bundles/niacubilla-v1-yolov8.zip'),
    'coffee_standard_v8': Path('/content/drive/MyDrive/Coffee_Bean_Detection/bundles/coffee-detection-with-standard-v8-yolov8.tar'),
}
for code,path in {**ROOTS,**ARCHIVES}.items():
    if not path.exists(): raise FileNotFoundError(f'{code}: {path}')
for code,archive in ARCHIVES.items():
    if code not in ROOTS:
        ROOTS[code]=extract_audit_archive(archive,Path('/content/public-dataset-audit-inputs')/code)
print('INPUT PATHS VALID')


In [ ]:
REGISTRY=REPO/'configs/public_dataset_audit/v2_candidate_registry.yaml'
OUTPUT=Path('/content/drive/MyDrive/Coffee_Bean_Detection/evidence/public-dataset-v2-audit')
result=audit_public_dataset_registry(REGISTRY,OUTPUT,root_overrides=ROOTS,archive_overrides=ARCHIVES,near_threshold=4)
print('DECISION:',result['decision'])
print('AUDITED:',result['audited_dataset_count'],'/',result['dataset_count'])
print('ELIGIBLE EXACT-HASH LINEAGES:',result['eligible_lineage_count'])
print('POTENTIAL LINEAGES AFTER REVIEW:',result['potential_lineage_count'])
for row in result['datasets']:
    print(row['code'],row['status'],row.get('images',0),row.get('boxes',0),row.get('reasons',[]))
print('TRAINING AUTHORIZED:',result['training_authorized'])
print('SUMMARY:',OUTPUT/'public_dataset_eligibility_summary.json')


In [ ]:
import pandas as pd
from IPython.display import display
display(pd.DataFrame([{k:row.get(k) for k in ('code','status','images','boxes','class_count','estimated_source_parents','reasons')} for row in result['datasets']]))
display(pd.DataFrame(result['cross_dataset']['near_candidates_by_dataset_pair']))
print('Kirim tabel dan decision. Jangan training.')
